# Redes Neuronales: La base del Deep Learning

En este notebook entenderemos cómo funcionan las redes neuronales — la tecnología detrás de la IA moderna.

## ¿Qué es una neurona artificial?

Una neurona artificial es una función matemática simple:

$$y = f\left(\sum_{i=1}^{n} w_i \cdot x_i + b\right)$$

Donde:
- $x_i$ = entradas (ej: área, pisos, zona sísmica)
- $w_i$ = pesos (importancia de cada entrada)
- $b$ = sesgo (bias)
- $f$ = función de activación (introduce no-linealidad)

### Analogía AEC
Piensa en la neurona como un **evaluador de riesgo sísmico**:
- Las entradas son las características del edificio
- Los pesos representan qué tan importante es cada característica
- La salida es el nivel de riesgo

```
  Área (m²) ──── w₁ ──→ ┐
  Pisos     ──── w₂ ──→ │ Σ + b → f(·) → Salida
  Zona      ──── w₃ ──→ ┘
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Visualizar funciones de activación comunes ---
x = np.linspace(-5, 5, 200)

# Funciones de activación
sigmoid = 1 / (1 + np.exp(-x))
relu = np.maximum(0, x)
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, y, nombre, color in zip(
    axes,
    [sigmoid, relu, tanh],
    ['Sigmoid', 'ReLU', 'Tanh'],
    ['#e74c3c', '#2ecc71', '#3498db']
):
    ax.plot(x, y, color=color, linewidth=2.5)
    ax.set_title(nombre, fontsize=14)
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('Entrada')
    ax.set_ylabel('Salida')

plt.suptitle('Funciones de Activación', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## De una neurona a una red

Una **red neuronal** es simplemente muchas neuronas conectadas en capas:

```
CAPA DE ENTRADA     CAPA OCULTA        CAPA DE SALIDA
                                        
  ○ (área)     ──→   ● ──┐             
                     ● ──┤──→  ● (costo estimado)
  ○ (pisos)    ──→   ● ──┤             
                     ● ──┘             
  ○ (material) ──→   ●                 
```

Cada conexión tiene un **peso** que el modelo ajusta durante el entrenamiento.

## ¿Cómo aprende una red neuronal?

El proceso de entrenamiento tiene 3 pasos que se repiten miles de veces:

### 1. Forward Pass (Propagación hacia adelante)
Los datos entran por la capa de entrada y viajan hasta la salida.

### 2. Cálculo del Error (Loss)
Se compara la predicción con el valor real. La diferencia es el **error**.

### 3. Backpropagation (Propagación hacia atrás)
El error se propaga hacia atrás para ajustar los pesos, reduciendo el error en la siguiente iteración.

```
Datos → [Red Neuronal] → Predicción
                              │
                    Comparar con valor real
                              │
                         Calcular error
                              │
                   Ajustar pesos (backprop)
                              │
                         Repetir ×1000
```

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Ejemplo: Red neuronal para estimar costo de obra ---
np.random.seed(42)
n = 200

# Características: área (m²), pisos, % zona sísmica alta (0-1)
area = np.random.uniform(100, 3000, n)
pisos = np.random.randint(1, 30, n)
zona_sismica = np.random.uniform(0, 1, n)

# Costo real (relación no lineal)
costo = (area * 2.5 
         + pisos ** 1.3 * 50 
         + zona_sismica * area * 0.3 
         + np.random.normal(0, 100, n))

X = np.column_stack([area, pisos, zona_sismica])
y = costo

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar (muy importante para redes neuronales)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)
y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

# Entrenar red neuronal
red = MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
red.fit(X_train_s, y_train_s)

# Predecir y des-escalar
y_pred_s = red.predict(X_test_s)
y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()

# Visualización
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=60)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lim, lim, 'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Costo real (millones COP)')
ax.set_ylabel('Costo predicho (millones COP)')
ax.set_title('Red Neuronal — Estimación de Costo de Obra')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Métricas
from sklearn.metrics import mean_absolute_error, r2_score
print(f"Error absoluto medio: {mean_absolute_error(y_test, y_pred):.0f} millones COP")
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")

## Tipos de redes neuronales relevantes para AEC

| Tipo | Uso principal | Aplicación AEC |
|------|--------------|----------------|
| **MLP** (Perceptrón multicapa) | Datos tabulares | Estimación de costos, predicción de plazos |
| **CNN** (Convolucional) | Imágenes | Detección de grietas, clasificación de materiales |
| **RNN/LSTM** (Recurrente) | Secuencias temporales | Predicción de avance de obra, series de tiempo de sensores |
| **Transformer** | Texto, secuencias | LLMs para documentación técnica, chatbots de obra |
| **GNN** (Grafos) | Datos en red | Análisis de modelos BIM (elementos conectados) |

## Ejercicio propuesto

1. Cambia `hidden_layer_sizes` a `(64, 32, 16)` — una red más profunda. ¿Mejora el R²?
2. Prueba con `max_iter=2000`. ¿Cambia el resultado?
3. Quita la variable `zona_sismica` del entrenamiento. ¿Cuánto empeora la predicción?